In [ ]:
import datetime as dt
import os

import xarray as xr
from pyproj import CRS

In [ ]:
from etdataset.dem import get_dem_from_roi
from etdataset.era5 import ERA5Dataset, ERA5Var
from etdataset.icos import get_stations_config
from etdataset.utils import work_area_from_coord_point
from etdataset.validation_temp.temperature_rescaling import (
    generate_datetime,
    get_era5_grid,
    get_ta_td_celsius_at_location,
    prepare_temperature_inputs,
    read_era5_file,
    read_era5_files,
    save_ta_td_csv,
    temperature_rescaling_constant_lapse_rate,
)

# Comparison temperature rescaling

## Prepare inputs

### Generate a list of datetimes for the desired time period

In [ ]:
list_date = generate_datetime(
    dt.date(2023, 3, 3),
    dt.date(2023, 3, 4),
    day_step=1,
    hour_start=2,
    hour_end=20,
    hour_step=2,
)

In [ ]:
list_date

### Read one ERA5 file

In [ ]:
era5_xrds = read_era5_file(
    dt.datetime(2023, 3, 3, 0, 0), ERA5Dataset.ERA5, "out"
)

### Read all ERA5 files for the dates in the list

In [ ]:
era5_xrds = read_era5_files(list_date, "out")

In [ ]:
era5_xrds

### Get metadatas of a desired station

In [ ]:
cfg = get_stations_config("DE-Geb")

In [ ]:
cfg

## Extract air temperature (TA) and dew point temperature (TD) at the station location from ERA5 data 

### Get a ROI around the station

In [ ]:
bounds, crs_bounds = work_area_from_coord_point(
    cfg.lat, cfg.lon, 50000, 50000, CRS.from_epsg(4326)
)["lat/lon"]


### Create an ERA5 grid over the ROI

In [ ]:
grid = get_era5_grid(bounds, crs_bounds, 0.25)

### Prepare the grid and era5 data for the temperature rescaling

In [ ]:
dem, era5_dem, updated_data = prepare_temperature_inputs(
    data=grid,
    era5_data=era5_xrds,
    dataset=ERA5Dataset.ERA5,
)

### ERA5 temperature rescaling using a constant lapse rate. 

The temperatures were computed at a given level and then adjusted to a reference level.

In [ ]:
updated = temperature_rescaling_constant_lapse_rate(
    updated_data,
    dem,
    era5_xrds,
    era5_dem,
    variables=[
        ERA5Var.TEMPERATURE,
        ERA5Var.DEWPOINT_TEMPERATURE,
    ],
)

In [ ]:
updated

### Extract air temperature (Ta) and dew point temperature (Td) in Celsius at station's location

In [ ]:
ta, td = get_ta_td_celsius_at_location(updated, cfg.lon, cfg.lat)

In [ ]:
ta

In [ ]:
td

# Extract TA and TD at the station location from DEM-adjusted ERA5 temperatures.

### Get a ROI around the station

In [ ]:
roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
    cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
)["utm"]


In [ ]:
mnt_path = os.environ["MNT_PATH"]

### Get a DEM over the ROI

In [ ]:
dem = get_dem_from_roi(
    roi_bbox=roi_bbox_utm,
    roi_crs=roi_crs_utm,
    base_dir=mnt_path,
    resolution=60,
)

In [ ]:
dem

### Prepare datas for the temperature rescaling

In [ ]:
dem, era5_dem, updated_data = prepare_temperature_inputs(
    data=dem,
    era5_data=era5_xrds,
    dataset=ERA5Dataset.ERA5,
)


### Adjust ERA5 temperatures (air and dew point) to the DEM using a constant lapse rate.

In [ ]:
updated = temperature_rescaling_constant_lapse_rate(
    updated_data,
    dem,
    era5_xrds,
    era5_dem,
    variables=[
        ERA5Var.TEMPERATURE,
        ERA5Var.DEWPOINT_TEMPERATURE,
    ],
)


In [ ]:
updated

Get station's location in UTM

In [ ]:
x, y = (
    work_area_from_coord_point(cfg.lat, cfg.lon, 0, 0, CRS.from_epsg(4326))[
        "utm"
    ][0].left,
    work_area_from_coord_point(cfg.lat, cfg.lon, 0, 0, CRS.from_epsg(4326))[
        "utm"
    ][0].bottom,
)


### Extract air temperature (Ta) and dew point temperature (Td) in Celsius at station's location

In [ ]:
ta, td = get_ta_td_celsius_at_location(updated, x, y)


In [ ]:
ta

In [ ]:
td

In [ ]:
ds_out = xr.Dataset(
    {
        "ta": ta,
        "tdp": td,
    }
)


### Save air temperature (TA) and dew point temperature (TD) data to a CSV file.

In [ ]:
save_ta_td_csv(ds_out, cfg)